# Disaster LLM IRT + K-factor Analysis

把这个 notebook 放在 `pyrecode E6` 项目根目录运行。默认会读取：

```text
results/irt_baseline_ruleset/response_matrix.csv
results/irt_baseline_ruleset/case_metadata.csv
results/irt_baseline_ruleset/irt_1pl_item_parameters.csv
results/irt_baseline_ruleset/irt_1pl_subject_ability.csv
results/irt_baseline_ruleset/missing_response_summary.csv  # optional
```

输出会写到：

```text
analysis_outputs/irt_kfactor/
```

这个 notebook 的核心流程：

1. 读取 response matrix 和 metadata；
2. 计算 model performance / missingness / paired tests；
3. 分析 IRT model ability；
4. 验证 IRT difficulty 是否对应真实 case 难度；
5. 做 residual K-factor 分析；
6. 做 new-item cold-start：metadata → item difficulty/loading → probabilistic response prediction；
7. 保存 CSV 和图。

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Put this notebook in:
# G:/Other computers/My Computer/Research/ai/measurement/pyrecode E6
PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from disaster_irt.paths import default_paths, ensure_dir
from disaster_irt.data import load_core_tables, build_decision_matrix_from_metadata, save_tables
from disaster_irt.features import flatten_case_metadata
from disaster_irt.metrics import model_performance, paired_comparisons, vote_fraction_analysis
from disaster_irt.irt import (
    ability_table, item_difficulty_table, difficulty_decile_summary,
    damage_label_difficulty, residual_factor_analysis
)
from disaster_irt.validity import feature_label_model_scores, difficulty_validity_correlations
from disaster_irt.kfactor import run_cold_start_experiment, cold_start_one_split
from disaster_irt import plots

paths = default_paths(PROJECT_ROOT)
IRT_DIR = paths["irt_dir"]
OUTPUT_DIR = ensure_dir(paths["output_dir"])
FIG_DIR = ensure_dir(OUTPUT_DIR / "figures")
TABLE_DIR = ensure_dir(OUTPUT_DIR / "tables")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IRT_DIR:", IRT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 1. Load core tables

In [ ]:
data = load_core_tables(IRT_DIR)
response = data["response_matrix"]
meta = data["case_metadata"]
item_params = data["item_params"]
subject = data["subject_ability"]
missing_log = data["missing_summary"]

print(data["paths"].to_string(index=False))
print("
response matrix:", response.shape)
print("case metadata:", meta.shape)
print("item params:", item_params.shape)
print("subject ability:", subject.shape)
response.head()

## 2. Flatten metadata and compute standard performance

`flatten_case_metadata` 把 `household_attributes` 里的嵌套字段展开成：State, Tenure, Income, Occupants, BuildingDamage, WaterAccess, PowerAccess, FoodAccess, UnsanitaryConditions 等。后面的 difficulty validity 和 K-factor cold-start 都用这些输入特征。

In [ ]:
flat = flatten_case_metadata(meta)
perf = model_performance(response, meta)
paired = paired_comparisons(response)

flat.to_csv(TABLE_DIR / "flattened_case_metadata.csv", index=False)
perf.to_csv(TABLE_DIR / "model_performance.csv", index=False)
paired.to_csv(TABLE_DIR / "paired_comparisons.csv", index=False)

perf.sort_values("accuracy", ascending=False).head(10)

In [ ]:
paired

## 3. IRT model ability

这里直接使用 `irt_1pl_subject_ability.csv` 的 1PL/Rasch ability，然后和 accuracy/coverage/recall 合并。

In [ ]:
ability = ability_table(subject, perf)
ability.to_csv(TABLE_DIR / "irt_ability_with_performance.csv", index=False)
ability.head(18)

In [ ]:
plots.plot_ability(ability, FIG_DIR / "irt_ability.png")
plots.plot_accuracy_vs_ability(ability, FIG_DIR / "accuracy_vs_ability.png")
plots.plot_coverage(perf, FIG_DIR / "coverage.png")
print("saved ability / accuracy / coverage plots")

## 4. IRT item difficulty validity

这部分是 IRT 的核心：我们不只说哪些 item 错得多，而是测试 IRT difficulty 是否对应真实的 case-level behavioral complexity。

步骤：

1. 用 metadata 训练一个 feature-only label model，预测真实 binary displacement label；
2. 用 cross-fitted probability 构造：
   - `feature_label_surprisal = -log P(true label | features)`；
   - `feature_ambiguity = 1 - |2p - 1|`；
3. 用 simple cue rules 构造：
   - severe cues but short label；
   - weak cues but long label；
   - cue-label conflict；
4. 看这些变量和 IRT difficulty 的 Spearman correlation。

In [ ]:
scored_meta, feature_metrics = feature_label_model_scores(flat, n_splits=5, random_state=2026)
item_table = item_difficulty_table(item_params, scored_meta)
validity_corr = difficulty_validity_correlations(item_table)
deciles = difficulty_decile_summary(item_table)
damage_label = damage_label_difficulty(item_table)

pd.DataFrame([feature_metrics]).to_csv(TABLE_DIR / "feature_label_model_metrics.csv", index=False)
item_table.to_csv(TABLE_DIR / "item_table_with_validity_features.csv", index=False)
validity_corr.to_csv(TABLE_DIR / "difficulty_validity_correlations.csv", index=False)
deciles.to_csv(TABLE_DIR / "difficulty_decile_summary.csv", index=False)
damage_label.to_csv(TABLE_DIR / "damage_label_difficulty.csv", index=False)

print(feature_metrics)
validity_corr

In [ ]:
damage_label.head(20)

In [ ]:
deciles

In [ ]:
plots.plot_validity_correlations(validity_corr, FIG_DIR / "difficulty_validity_correlations.png")
plots.plot_damage_label_difficulty(damage_label, FIG_DIR / "damage_label_difficulty.png")
plots.plot_difficulty_deciles(deciles, FIG_DIR / "difficulty_deciles.png")
print("saved difficulty validity plots")

## 5. Cross-model vote fraction as uncertainty signal

这个不是 IRT 本身，但可以和 IRT/K-factor 的 probability story 连接起来：多个模型预测 long 的比例越高，真实 long rate 应该越高。

In [ ]:
decision = build_decision_matrix_from_metadata(meta, model_names=response["model"])
vote_summary = vote_fraction_analysis(decision, meta)
vote_summary.to_csv(TABLE_DIR / "vote_fraction_summary.csv", index=False)
plots.plot_vote_fraction(vote_summary, FIG_DIR / "vote_fraction_calibration.png")
vote_summary

## 6. Residual K-factor analysis

1PL/Rasch 假设所有模型差异都可以压缩成单一 ability，所有 item 差异都可以压缩成单一 difficulty。Residual factor analysis 用来检查这个假设是否足够。

如果 factor 1 的 subject loading 和 predicted-long rate / ruleset prompt 强相关，就说明模型间还有一个 “long-displacement threshold” 维度。

In [ ]:
residual = residual_factor_analysis(response, subject, item_params, n_factors=5)
subject_factors = residual["subject_factors"]
item_factors = residual["item_factors"]
factor_variance = residual["variance"]

subject_factors.to_csv(TABLE_DIR / "residual_factor_subject_loadings.csv", index=False)
item_factors.to_csv(TABLE_DIR / "residual_factor_item_loadings.csv", index=False)
factor_variance.to_csv(TABLE_DIR / "residual_factor_variance.csv", index=False)

plots.plot_residual_factor_variance(factor_variance, FIG_DIR / "residual_factor_variance.png")
plots.plot_subject_factor(subject_factors, FIG_DIR / "residual_factor1_subject_loadings.png", factor="factor1")

factor_variance

In [ ]:
subject_factors.sort_values("factor1", ascending=False)

## 7. K-factor new-item cold-start: correctness response

这里测试你说的 pipeline：

```text
mask 几百个 item
用剩下 item 拟合 K-factor
得到 train item 的 difficulty/loading
训练 metadata -> difficulty/loading
对 masked item 预测 loading
再用概率模型得到 P(model m on item i is correct)
```

这个版本的 response 是 IRT correctness：`Y_mi=1` 表示模型答对。

In [ ]:
RUN_KFACTOR = True
SEEDS = [2026, 2027, 2028, 2029, 2030]
KS = [0, 1, 2, 4, 8]
TEST_SIZE = 500

if RUN_KFACTOR:
    correctness_cold = run_cold_start_experiment(
        response_matrix=response,
        flat_metadata=flat,
        seeds=SEEDS,
        Ks=KS,
        test_size=TEST_SIZE,
        alpha=10.0,
        min_observed_models=10,
    )
    correctness_cold["summary"].to_csv(TABLE_DIR / "kfactor_correctness_coldstart_summary.csv", index=False)
    correctness_cold["metrics_by_split"].to_csv(TABLE_DIR / "kfactor_correctness_coldstart_by_split.csv", index=False)
    correctness_cold["diagnostics_by_split"].to_csv(TABLE_DIR / "kfactor_correctness_coldstart_diagnostics_by_split.csv", index=False)
    correctness_cold["diagnostics_summary"].to_csv(TABLE_DIR / "kfactor_correctness_coldstart_diagnostics_summary.csv", index=False)
    plots.plot_coldstart_auc(correctness_cold["summary"], FIG_DIR / "kfactor_correctness_coldstart_auc.png", "Correctness cold-start: metadata → item loading")
    plots.plot_coldstart_brier(correctness_cold["summary"], FIG_DIR / "kfactor_correctness_coldstart_brier.png", "Correctness cold-start: metadata → item loading")
    display(correctness_cold["summary"])
else:
    print("K-factor experiment skipped")

## 8. K-factor new-item cold-start: raw decision response

这个版本不预测 correctness，而是预测：

```text
Y_mi=1 if model m predicts LeaveHomeForMoreThanAWeek
```

所以输出可以解释成：

```text
P(model m predicts Long | new item features)
```

这个更接近 deployment 时的 probabilistic response surrogate。

In [ ]:
if RUN_KFACTOR:
    decision_cold = run_cold_start_experiment(
        response_matrix=decision,
        flat_metadata=flat,
        seeds=SEEDS,
        Ks=KS,
        test_size=TEST_SIZE,
        alpha=10.0,
        min_observed_models=10,
    )
    decision_cold["summary"].to_csv(TABLE_DIR / "kfactor_decision_coldstart_summary.csv", index=False)
    decision_cold["metrics_by_split"].to_csv(TABLE_DIR / "kfactor_decision_coldstart_by_split.csv", index=False)
    decision_cold["diagnostics_by_split"].to_csv(TABLE_DIR / "kfactor_decision_coldstart_diagnostics_by_split.csv", index=False)
    decision_cold["diagnostics_summary"].to_csv(TABLE_DIR / "kfactor_decision_coldstart_diagnostics_summary.csv", index=False)
    plots.plot_coldstart_auc(decision_cold["summary"], FIG_DIR / "kfactor_decision_coldstart_auc.png", "Raw decision cold-start: metadata → item loading")
    plots.plot_coldstart_brier(decision_cold["summary"], FIG_DIR / "kfactor_decision_coldstart_brier.png", "Raw decision cold-start: metadata → item loading")
    display(decision_cold["summary"])
else:
    print("K-factor experiment skipped")

## 9. Save one detailed held-out split for inspection

这个输出可以用来检查某个 masked item 上每个模型的预测概率。

In [ ]:
if RUN_KFACTOR:
    m4, d4, preds4 = cold_start_one_split(response, flat, seed=2026, test_size=500, K=4, alpha=10.0, return_predictions=True)
    preds4.to_csv(TABLE_DIR / "heldout_predictions_correctness_seed2026_K4.csv", index=False)
    display(preds4.head())
    print("Detailed K=4 correctness predictions saved.")

## 10. Minimal interpretation helpers

这几个表通常最适合直接放进 report 或 LaTeX：

- `irt_ability_with_performance.csv`
- `difficulty_validity_correlations.csv`
- `damage_label_difficulty.csv`
- `difficulty_decile_summary.csv`
- `residual_factor_variance.csv`
- `kfactor_correctness_coldstart_summary.csv`
- `kfactor_decision_coldstart_summary.csv`

图在 `analysis_outputs/irt_kfactor/figures/`。

In [ ]:
print("All outputs saved under:", OUTPUT_DIR)
print("Tables:", TABLE_DIR)
print("Figures:", FIG_DIR)
print("
Top output files:")
for p in sorted(TABLE_DIR.glob("*.csv"))[:30]:
    print(" -", p.name)